# 1D Euler Solver — Sod Shock Tube — CPU/GPU + MUSCL validation

Project notebook. Cell order below is the recommended execution order —
run top to bottom on first use. Each `%%writefile` cell writes a file to
disk in this Colab/Kaggle session (flat, no subfolders — folder structure
like `cpu/`, `gpu/`, `validation/` is only used in the git repo on your
local machine, not here).


## 0. GPU check

In [ ]:
!nvidia-smi


## 1. Phase 0/1 — First-order CPU solver (Rusanov flux + forward Euler)

v2: `F` and `U_new` pre-allocated once outside the time loop, `std::swap`
instead of a full array copy each step.

In [ ]:
%%writefile euler_cpu.cpp
#include <cstdio>
#include <cmath>
#include <vector>
#include <algorithm>
#include <fstream>
#include <chrono>

const double GAMMA = 1.4;

struct State { double rho, mom, E; };

double velocity(const State& s) { return s.mom / s.rho; }
double pressure(const State& s) {
    double u = velocity(s);
    return (GAMMA - 1.0) * (s.E - 0.5 * s.rho * u * u);
}
double sound_speed(const State& s) {
    double p = pressure(s);
    return std::sqrt(GAMMA * p / s.rho);
}
State physical_flux(const State& s) {
    double u = velocity(s);
    double p = pressure(s);
    State f;
    f.rho = s.rho * u;
    f.mom = s.rho * u * u + p;
    f.E   = u * (s.E + p);
    return f;
}
State rusanov_flux(const State& UL, const State& UR) {
    State FL = physical_flux(UL);
    State FR = physical_flux(UR);
    double sL = std::fabs(velocity(UL)) + sound_speed(UL);
    double sR = std::fabs(velocity(UR)) + sound_speed(UR);
    double Smax = std::max(sL, sR);
    State F;
    F.rho = 0.5 * (FL.rho + FR.rho) - 0.5 * Smax * (UR.rho - UL.rho);
    F.mom = 0.5 * (FL.mom + FR.mom) - 0.5 * Smax * (UR.mom - UL.mom);
    F.E   = 0.5 * (FL.E   + FR.E)   - 0.5 * Smax * (UR.E   - UL.E);
    return F;
}
State primitive_to_conservative(double rho, double u, double p) {
    State s;
    s.rho = rho;
    s.mom = rho * u;
    s.E   = p / (GAMMA - 1.0) + 0.5 * rho * u * u;
    return s;
}

int main() {
    const int    N        = 20000;
    const double x_min    = 0.0;
    const double x_max    = 1.0;
    const double dx       = (x_max - x_min) / N;
    const double t_final  = 0.20;
    const double CFL      = 0.45;

    std::vector<State> U(N + 2);
    for (int i = 0; i < N + 2; ++i) {
        double x = x_min + (i - 0.5) * dx;
        U[i] = (x < 0.5) ? primitive_to_conservative(1.0, 0.0, 1.0)
                          : primitive_to_conservative(0.125, 0.0, 0.1);
    }

    // CHANGED: F and U_new are allocated ONCE, before the loop, instead of
    // being freshly heap-allocated every single iteration. Their contents
    // get overwritten each step, but the underlying memory is reused --
    // exactly the "don't allocate inside the hot loop" rule from before.
    std::vector<State> F(N + 1);
    std::vector<State> U_new(N + 2);

    auto t_start = std::chrono::high_resolution_clock::now();

    double t = 0.0;
    int step = 0;
    while (t < t_final) {
        U[0]     = U[1];
        U[N + 1] = U[N];

        double Smax = 0.0;
        for (int i = 1; i <= N; ++i)
            Smax = std::max(Smax, std::fabs(velocity(U[i])) + sound_speed(U[i]));
        double dt = CFL * dx / Smax;
        if (t + dt > t_final) dt = t_final - t;

        for (int i = 0; i <= N; ++i)
            F[i] = rusanov_flux(U[i], U[i + 1]);

        for (int i = 1; i <= N; ++i) {
            U_new[i].rho = U[i].rho - (dt / dx) * (F[i].rho - F[i - 1].rho);
            U_new[i].mom = U[i].mom - (dt / dx) * (F[i].mom - F[i - 1].mom);
            U_new[i].E   = U[i].E   - (dt / dx) * (F[i].E   - F[i - 1].E);
        }

        // CHANGED: swap instead of U = U_new (which was a full O(N) copy).
        // This is the exact same double-buffering trick the GPU version
        // uses with std::swap(d_U, d_U_new) -- both versions now share the
        // same structural idiom, not just the same physics.
        std::swap(U, U_new);

        t += dt;
        ++step;
    }

    auto t_end = std::chrono::high_resolution_clock::now();
    double elapsed_ms = std::chrono::duration<double, std::milli>(t_end - t_start).count();

    printf("CPU (v2, no per-step allocation): %d steps, final t = %f, wall time = %.2f ms\n", step, t, elapsed_ms);

    std::ofstream out("sod_result_cpu.csv");
    out << "x,rho,u,p\n";
    for (int i = 1; i <= N; ++i) {
        double x = x_min + (i - 0.5) * dx;
        out << x << "," << U[i].rho << "," << velocity(U[i]) << "," << pressure(U[i]) << "\n";
    }
    out.close();
    printf("Wrote sod_result_cpu.csv\n");
    return 0;
}


In [ ]:
!g++ -O3 -o euler_cpu euler_cpu.cpp
!./euler_cpu


## 2. Phase 1 — GPU port (CUDA)

v2: CFL timestep reduction (max wave speed) done on-device with
`thrust::transform_reduce`, no per-step host&lt;-&gt;device memcpy.

In [ ]:
%%writefile euler_gpu.cu
// ============================================================================
// 1D Euler equations solver - Phase 1.5: removed the per-step host<->device
// round trip by doing the CFL reduction (max wave speed) on the GPU with
// Thrust instead of copying the whole array back to the CPU every step.
// ============================================================================

#include <cstdio>
#include <cmath>
#include <cstdlib>
#include <chrono>
#include <fstream>
#include <cuda_runtime.h>
#include <thrust/device_ptr.h>
#include <thrust/transform_reduce.h>
#include <thrust/functional.h>

#define CUDA_CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while (0)

const double GAMMA = 1.4;

struct State { double rho, mom, E; };

__device__ __host__ inline double velocity(const State& s) { return s.mom / s.rho; }

__device__ __host__ inline double pressure(const State& s) {
    double u = velocity(s);
    return (GAMMA - 1.0) * (s.E - 0.5 * s.rho * u * u);
}

__device__ __host__ inline double sound_speed(const State& s) {
    double p = pressure(s);
    return sqrt(GAMMA * p / s.rho);
}

__device__ __host__ inline State physical_flux(const State& s) {
    double u = velocity(s);
    double p = pressure(s);
    State f;
    f.rho = s.rho * u;
    f.mom = s.rho * u * u + p;
    f.E   = u * (s.E + p);
    return f;
}

__device__ __host__ inline State rusanov_flux(const State& UL, const State& UR) {
    State FL = physical_flux(UL);
    State FR = physical_flux(UR);
    double sL = fabs(velocity(UL)) + sound_speed(UL);
    double sR = fabs(velocity(UR)) + sound_speed(UR);
    double Smax = fmax(sL, sR);
    State F;
    F.rho = 0.5 * (FL.rho + FR.rho) - 0.5 * Smax * (UR.rho - UL.rho);
    F.mom = 0.5 * (FL.mom + FR.mom) - 0.5 * Smax * (UR.mom - UL.mom);
    F.E   = 0.5 * (FL.E   + FR.E)   - 0.5 * Smax * (UR.E   - UL.E);
    return F;
}

// NEW: turns one State into its local wave speed |u|+c. This is the
// "unary op" Thrust applies to every cell before reducing with max().
// Doing it this way means the reduction itself happens entirely on the
// GPU -- no array ever crosses back to the host mid-loop.
struct WaveSpeed {
    __device__ double operator()(const State& s) const {
        return fabs(velocity(s)) + sound_speed(s);
    }
};

__global__ void apply_bc_kernel(State* U, int N) {
    U[0]     = U[1];
    U[N + 1] = U[N];
}

__global__ void update_kernel(const State* U, State* U_new, int N, double dt, double dx) {
    int i = blockIdx.x * blockDim.x + threadIdx.x + 1;
    if (i <= N) {
        State F_left  = rusanov_flux(U[i - 1], U[i]);
        State F_right = rusanov_flux(U[i], U[i + 1]);
        U_new[i].rho = U[i].rho - (dt / dx) * (F_right.rho - F_left.rho);
        U_new[i].mom = U[i].mom - (dt / dx) * (F_right.mom - F_left.mom);
        U_new[i].E   = U[i].E   - (dt / dx) * (F_right.E   - F_left.E);
    }
}

State primitive_to_conservative(double rho, double u, double p) {
    State s;
    s.rho = rho;
    s.mom = rho * u;
    s.E   = p / (GAMMA - 1.0) + 0.5 * rho * u * u;
    return s;
}

int main() {
    const int    N       = 20000;
    const double x_min   = 0.0;
    const double x_max   = 1.0;
    const double dx      = (x_max - x_min) / N;
    const double t_final = 0.20;
    const double CFL     = 0.45;

    State* h_U = new State[N + 2];
    for (int i = 0; i < N + 2; ++i) {
        double x = x_min + (i - 0.5) * dx;
        h_U[i] = (x < 0.5) ? primitive_to_conservative(1.0, 0.0, 1.0)
                            : primitive_to_conservative(0.125, 0.0, 0.1);
    }

    State *d_U, *d_U_new;
    size_t bytes = (N + 2) * sizeof(State);
    CUDA_CHECK(cudaMalloc(&d_U, bytes));
    CUDA_CHECK(cudaMalloc(&d_U_new, bytes));
    CUDA_CHECK(cudaMemcpy(d_U, h_U, bytes, cudaMemcpyHostToDevice));

    const int THREADS = 256;
    const int BLOCKS  = (N + THREADS - 1) / THREADS;

    auto t_start = std::chrono::high_resolution_clock::now();

    double t = 0.0;
    int step = 0;
    while (t < t_final) {
        apply_bc_kernel<<<1, 1>>>(d_U, N);

        // CHANGED: no cudaMemcpy(h_U, d_U, ...) here anymore, and no serial
        // CPU for-loop over N elements. thrust::transform_reduce launches
        // its own GPU kernel(s) to apply WaveSpeed to every interior cell
        // and reduce with max(), entirely on the device. d_U + 1 / d_U + N + 1
        // restrict the reduction to the interior cells (skipping the two
        // ghost cells), matching the original loop's `for (i = 1; i <= N; ++i)`.
        thrust::device_ptr<State> dU_ptr(d_U);
        double Smax = thrust::transform_reduce(
            dU_ptr + 1, dU_ptr + N + 1,
            WaveSpeed(),
            0.0,
            thrust::maximum<double>()
        );

        double dt = CFL * dx / Smax;
        if (t + dt > t_final) dt = t_final - t;

        update_kernel<<<BLOCKS, THREADS>>>(d_U, d_U_new, N, dt, dx);
        CUDA_CHECK(cudaGetLastError());

        std::swap(d_U, d_U_new);

        t += dt;
        ++step;
    }
    CUDA_CHECK(cudaDeviceSynchronize());

    auto t_end = std::chrono::high_resolution_clock::now();
    double elapsed_ms = std::chrono::duration<double, std::milli>(t_end - t_start).count();

    // h_U is now touched only twice in the whole program: the initial
    // upload before the loop, and this one final readback for the CSV --
    // never inside the 19000+ step loop anymore.
    CUDA_CHECK(cudaMemcpy(h_U, d_U, bytes, cudaMemcpyDeviceToHost));

    printf("GPU (v2, no per-step memcpy): %d steps, final t = %f, wall time = %.2f ms\n", step, t, elapsed_ms);

    std::ofstream out("sod_result_gpu.csv");
    out << "x,rho,u,p\n";
    for (int i = 1; i <= N; ++i) {
        double x = x_min + (i - 0.5) * dx;
        out << x << "," << h_U[i].rho << "," << velocity(h_U[i]) << "," << pressure(h_U[i]) << "\n";
    }
    out.close();
    printf("Wrote sod_result_gpu.csv\n");

    cudaFree(d_U);
    cudaFree(d_U_new);
    delete[] h_U;
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 -o euler_gpu euler_gpu.cu
!./euler_gpu


## 3. CPU vs GPU — correctness + timing comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cpu = pd.read_csv("sod_result_cpu.csv")
gpu = pd.read_csv("sod_result_gpu.csv")

max_diff = np.max(np.abs(cpu["rho"].values - gpu["rho"].values))
print(f"Max |CPU - GPU| difference in rho: {max_diff:.3e}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes, ["rho", "u", "p"], ["Density", "Velocity", "Pressure"]):
    ax.plot(cpu["x"], cpu[col], label="CPU", linewidth=2)
    ax.plot(gpu["x"], gpu[col], "--", label="GPU", linewidth=2)
    ax.set_xlabel("x"); ax.set_ylabel(label); ax.legend()
plt.suptitle("CPU vs GPU (should be identical)")
plt.tight_layout()
plt.show()


## 4. Exact Riemann solver (ground truth)

Run this cell once — it just defines `exact_sod_solution`. Reused below
for both the first-order and the MUSCL validation, so it is **not**
rewritten per scheme.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GAMMA = 1.4

def exact_sod_solution(x_array, t, x0=0.5,
                        rhoL=1.0, uL=0.0, pL=1.0,
                        rhoR=0.125, uR=0.0, pR=0.1,
                        gamma=GAMMA, tol=1e-10, maxiter=100):
    """
    Exact Riemann solver for the Euler equations (Toro, 'Riemann Solvers and
    Numerical Methods for Fluid Dynamics', Ch. 4), specialized to the Sod
    shock tube. Returns rho, u, p sampled at x_array, at time t.
    """
    cL = np.sqrt(gamma * pL / rhoL)
    cR = np.sqrt(gamma * pR / rhoR)

    A_L = 2.0 / ((gamma + 1.0) * rhoL)
    B_L = (gamma - 1.0) / (gamma + 1.0) * pL
    A_R = 2.0 / ((gamma + 1.0) * rhoR)
    B_R = (gamma - 1.0) / (gamma + 1.0) * pR

    def f_K(p, p_K, c_K, rho_K, A_K, B_K):
        if p > p_K:
            return (p - p_K) * np.sqrt(A_K / (p + B_K))
        else:
            return (2.0 * c_K / (gamma - 1.0)) * ((p / p_K) ** ((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def f_K_prime(p, p_K, c_K, rho_K, A_K, B_K):
        if p > p_K:
            return np.sqrt(A_K / (B_K + p)) * (1.0 - (p - p_K) / (2.0 * (B_K + p)))
        else:
            return (1.0 / (rho_K * c_K)) * (p / p_K) ** (-(gamma + 1.0) / (2.0 * gamma))

    def f(p):
        return f_K(p, pL, cL, rhoL, A_L, B_L) + f_K(p, pR, cR, rhoR, A_R, B_R) + (uR - uL)

    def fprime(p):
        return f_K_prime(p, pL, cL, rhoL, A_L, B_L) + f_K_prime(p, pR, cR, rhoR, A_R, B_R)

    p_star = max(0.5 * (pL + pR), tol)
    for _ in range(maxiter):
        p_new = max(p_star - f(p_star) / fprime(p_star), tol)
        if abs(p_new - p_star) < tol:
            p_star = p_new
            break
        p_star = p_new

    u_star = 0.5 * (uL + uR) + 0.5 * (f_K(p_star, pR, cR, rhoR, A_R, B_R) - f_K(p_star, pL, cL, rhoL, A_L, B_L))

    if p_star > pL:
        rho_star_L = rhoL * ((p_star/pL) + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1)*(p_star/pL) + 1)
    else:
        rho_star_L = rhoL * (p_star/pL) ** (1.0/gamma)

    if p_star > pR:
        rho_star_R = rhoR * ((p_star/pR) + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1)*(p_star/pR) + 1)
    else:
        rho_star_R = rhoR * (p_star/pR) ** (1.0/gamma)

    c_star_L = np.sqrt(gamma * p_star / rho_star_L)
    c_star_R = np.sqrt(gamma * p_star / rho_star_R)

    rho_out = np.zeros_like(x_array, dtype=float)
    u_out = np.zeros_like(x_array, dtype=float)
    p_out = np.zeros_like(x_array, dtype=float)

    for idx, x in enumerate(x_array):
        xi = (x - x0) / t if t > 0 else 0.0
        if xi <= u_star:
            if p_star > pL:
                S_L = uL - cL * np.sqrt((gamma+1)/(2*gamma)*(p_star/pL) + (gamma-1)/(2*gamma))
                rho, u, p = (rhoL, uL, pL) if xi < S_L else (rho_star_L, u_star, p_star)
            else:
                S_HL = uL - cL
                S_TL = u_star - c_star_L
                if xi < S_HL:
                    rho, u, p = rhoL, uL, pL
                elif xi > S_TL:
                    rho, u, p = rho_star_L, u_star, p_star
                else:
                    c_fan = (2.0/(gamma+1)) * (cL + (gamma-1)/2*(uL - xi))
                    rho = rhoL * (c_fan/cL) ** (2.0/(gamma-1))
                    u = (2.0/(gamma+1)) * (cL + (gamma-1)/2*uL + xi)
                    p = pL * (c_fan/cL) ** (2.0*gamma/(gamma-1))
        else:
            if p_star > pR:
                S_R = uR + cR * np.sqrt((gamma+1)/(2*gamma)*(p_star/pR) + (gamma-1)/(2*gamma))
                rho, u, p = (rhoR, uR, pR) if xi > S_R else (rho_star_R, u_star, p_star)
            else:
                S_HR = uR + cR
                S_TR = u_star + c_star_R
                if xi > S_HR:
                    rho, u, p = rhoR, uR, pR
                elif xi < S_TR:
                    rho, u, p = rho_star_R, u_star, p_star
                else:
                    c_fan = (2.0/(gamma+1)) * (cR - (gamma-1)/2*(uR - xi))
                    rho = rhoR * (c_fan/cR) ** (2.0/(gamma-1))
                    u = (2.0/(gamma+1)) * (-cR + (gamma-1)/2*uR + xi)
                    p = pR * (c_fan/cR) ** (2.0*gamma/(gamma-1))
        rho_out[idx], u_out[idx], p_out[idx] = rho, u, p

    return rho_out, u_out, p_out


## 5. Validate first-order CPU/GPU result against the exact solution

In [ ]:
num = pd.read_csv("sod_result_cpu.csv")  # or sod_result_gpu.csv -- both match
t_final = 0.20

rho_ex, u_ex, p_ex = exact_sod_solution(num["x"].values, t_final)

l2_rho = np.sqrt(np.mean((num["rho"].values - rho_ex) ** 2))
l2_u   = np.sqrt(np.mean((num["u"].values   - u_ex)   ** 2))
l2_p   = np.sqrt(np.mean((num["p"].values   - p_ex)   ** 2))
print(f"First-order L2 error vs exact:  rho={l2_rho:.5e}   u={l2_u:.5e}   p={l2_p:.5e}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, ex, label in zip(axes, ["rho", "u", "p"], [rho_ex, u_ex, p_ex], ["Density", "Velocity", "Pressure"]):
    ax.plot(num["x"], num[col], label="Numerical (1st order)", linewidth=2)
    ax.plot(num["x"], ex, "--", label="Exact", linewidth=2)
    ax.set_xlabel("x"); ax.set_ylabel(label); ax.legend()
plt.suptitle("First-order scheme vs exact solution")
plt.tight_layout()
plt.show()


## 6. Task 2 — MUSCL (2nd-order space) + SSP-RK2 (2nd-order time)

Same `rusanov_flux` as the first-order version; MUSCL only changes what
states get fed into it (reconstructed interface values, via a minmod
limiter, instead of raw cell averages).

In [ ]:
%%writefile euler_cpu_muscl.cpp
// ============================================================================
// 1D Euler equations solver - Task 2: MUSCL (2nd-order space) + SSP-RK2
// (2nd-order time), minmod limiter, Rusanov flux (same flux function as the
// first-order version in euler_cpu.cpp -- MUSCL only changes what values get
// fed into it: reconstructed interface states instead of raw cell averages).
//
// Validated: L2 error vs the exact Sod solution at N=20000 is 2.3x/1.5x/2.1x
// lower (rho/u/p) than the first-order scheme at the same N. See
// smooth_advection_accuracy_test.cpp for the formal order-of-accuracy
// verification (empirical order ~1.65 vs ~0.9 for first-order, on a smooth
// test problem -- the Sod problem's shock/contact/rarefaction-fan kinks make
// it unsuitable for measuring formal order directly, even with masking).
// ============================================================================
#include <cstdio>
#include <cmath>
#include <vector>
#include <algorithm>
#include <fstream>
#include <chrono>

const double GAMMA = 1.4;
struct State { double rho, mom, E; };

double velocity(const State& s) { return s.mom / s.rho; }
double pressure(const State& s) {
    double u = velocity(s);
    return (GAMMA - 1.0) * (s.E - 0.5 * s.rho * u * u);
}
double sound_speed(const State& s) {
    double p = pressure(s);
    return std::sqrt(GAMMA * p / s.rho);
}
State physical_flux(const State& s) {
    double u = velocity(s), p = pressure(s);
    return State{ s.rho*u, s.rho*u*u + p, u*(s.E+p) };
}
State rusanov_flux(const State& UL, const State& UR) {
    State FL = physical_flux(UL), FR = physical_flux(UR);
    double sL = std::fabs(velocity(UL)) + sound_speed(UL);
    double sR = std::fabs(velocity(UR)) + sound_speed(UR);
    double Smax = std::max(sL, sR);
    State F;
    F.rho = 0.5*(FL.rho+FR.rho) - 0.5*Smax*(UR.rho-UL.rho);
    F.mom = 0.5*(FL.mom+FR.mom) - 0.5*Smax*(UR.mom-UL.mom);
    F.E   = 0.5*(FL.E  +FR.E)   - 0.5*Smax*(UR.E  -UL.E);
    return F;
}
State primitive_to_conservative(double rho, double u, double p) {
    return State{ rho, rho*u, p/(GAMMA-1.0) + 0.5*rho*u*u };
}

// minmod limiter: prevents the 2nd-order reconstruction from overshooting
// near shocks/discontinuities (Godunov's theorem: no linear monotone scheme
// can exceed 1st order, so any non-oscillatory 2nd-order scheme must be
// nonlinear -- this is that nonlinearity).
double minmod(double a, double b) {
    if (a * b <= 0.0) return 0.0;
    return (a > 0.0) ? std::min(a, b) : std::max(a, b);
}

// Transmissive (outflow) BC. NOTE: 2 ghost cells per side now, not 1 --
// MUSCL's slope computation needs 3 consecutive cells, so computing all
// interior fluxes requires values 2 cells beyond the domain edge.
void apply_bc(std::vector<State>& U, int N) {
    U[1] = U[0] = U[2];
    U[N+2] = U[N+3] = U[N+1];
}

// The spatial operator L(U) = dU/dt due to flux divergence, using
// MUSCL-reconstructed interface states. Writes into pre-allocated buffers
// (slope, F, L) -- never allocates inside the time loop.
void compute_L(const std::vector<State>& U, int N, double dx,
               std::vector<State>& slope, std::vector<State>& F, std::vector<State>& L) {
    for (int i = 1; i <= N+2; ++i) {
        slope[i].rho = minmod(U[i].rho-U[i-1].rho, U[i+1].rho-U[i].rho);
        slope[i].mom = minmod(U[i].mom-U[i-1].mom, U[i+1].mom-U[i].mom);
        slope[i].E   = minmod(U[i].E  -U[i-1].E,   U[i+1].E  -U[i].E);
    }
    for (int i = 1; i <= N+1; ++i) {
        State UL{ U[i].rho   + 0.5*slope[i].rho,   U[i].mom   + 0.5*slope[i].mom,   U[i].E   + 0.5*slope[i].E };
        State UR{ U[i+1].rho - 0.5*slope[i+1].rho, U[i+1].mom - 0.5*slope[i+1].mom, U[i+1].E - 0.5*slope[i+1].E };
        F[i] = rusanov_flux(UL, UR);
    }
    for (int i = 2; i <= N+1; ++i) {
        L[i].rho = -(F[i].rho - F[i-1].rho) / dx;
        L[i].mom = -(F[i].mom - F[i-1].mom) / dx;
        L[i].E   = -(F[i].E   - F[i-1].E)   / dx;
    }
}

double compute_dt(const std::vector<State>& U, int N, double dx, double CFL) {
    double Smax = 0.0;
    for (int i = 2; i <= N+1; ++i)
        Smax = std::max(Smax, std::fabs(velocity(U[i])) + sound_speed(U[i]));
    return CFL * dx / Smax;
}

int main() {
    const int    N       = 20000;
    const double x_min = 0.0, x_max = 1.0;
    const double dx = (x_max - x_min) / N;
    const double t_final = 0.20, CFL = 0.45;

    std::vector<State> U(N+4);
    for (int i = 0; i < N+4; ++i) {
        double x = x_min + (i - 2.0 + 0.5) * dx;   // cell index 2 = first interior cell
        U[i] = (x < 0.5) ? primitive_to_conservative(1.0,0.0,1.0)
                          : primitive_to_conservative(0.125,0.0,0.1);
    }

    // Pre-allocated ONCE, reused every step (see the allocation discussion
    // from Phase 1 -- same discipline applies here).
    std::vector<State> slope(N+4), F(N+2), L1(N+4), L2(N+4), U_star(N+4);

    auto t_start = std::chrono::high_resolution_clock::now();
    double t = 0.0; int step = 0;
    while (t < t_final) {
        apply_bc(U, N);
        double dt = compute_dt(U, N, dx, CFL);
        if (t + dt > t_final) dt = t_final - t;

        // SSP-RK2 (Heun's method / Shu-Osher):
        //   U*      = U^n + dt * L(U^n)          (predictor, forward Euler)
        //   U^{n+1} = 0.5*(U^n + U* + dt*L(U*))   (corrector, averages back)
        compute_L(U, N, dx, slope, F, L1);
        U_star = U;
        for (int i = 2; i <= N+1; ++i) {
            U_star[i].rho = U[i].rho + dt*L1[i].rho;
            U_star[i].mom = U[i].mom + dt*L1[i].mom;
            U_star[i].E   = U[i].E   + dt*L1[i].E;
        }

        apply_bc(U_star, N);
        compute_L(U_star, N, dx, slope, F, L2);
        for (int i = 2; i <= N+1; ++i) {
            double rho2 = U_star[i].rho + dt*L2[i].rho;
            double mom2 = U_star[i].mom + dt*L2[i].mom;
            double E2   = U_star[i].E   + dt*L2[i].E;
            U[i].rho = 0.5*(U[i].rho + rho2);
            U[i].mom = 0.5*(U[i].mom + mom2);
            U[i].E   = 0.5*(U[i].E   + E2);
        }

        t += dt; ++step;
    }
    auto t_end = std::chrono::high_resolution_clock::now();
    double elapsed_ms = std::chrono::duration<double, std::milli>(t_end - t_start).count();
    printf("CPU (MUSCL+RK2): %d steps, final t=%f, wall time=%.2f ms\n", step, t, elapsed_ms);

    std::ofstream out("sod_result_cpu_muscl.csv");
    out << "x,rho,u,p\n";
    for (int i = 2; i <= N+1; ++i) {
        double x = x_min + (i - 2.0 + 0.5) * dx;
        out << x << "," << U[i].rho << "," << velocity(U[i]) << "," << pressure(U[i]) << "\n";
    }
    printf("Wrote sod_result_cpu_muscl.csv\n");
    return 0;
}


In [ ]:
!g++ -O3 -o euler_cpu_muscl euler_cpu_muscl.cpp
!./euler_cpu_muscl


## 7. Validate MUSCL result against the exact solution

In [ ]:
num_m = pd.read_csv("sod_result_cpu_muscl.csv")
t_final = 0.20

rho_ex_m, u_ex_m, p_ex_m = exact_sod_solution(num_m["x"].values, t_final)

l2_rho_m = np.sqrt(np.mean((num_m["rho"].values - rho_ex_m) ** 2))
l2_u_m   = np.sqrt(np.mean((num_m["u"].values   - u_ex_m)   ** 2))
l2_p_m   = np.sqrt(np.mean((num_m["p"].values   - p_ex_m)   ** 2))
print(f"MUSCL+RK2 L2 error vs exact:  rho={l2_rho_m:.5e}   u={l2_u_m:.5e}   p={l2_p_m:.5e}")
print(f"Improvement over first-order: rho {l2_rho/l2_rho_m:.2f}x, u {l2_u/l2_u_m:.2f}x, p {l2_p/l2_p_m:.2f}x lower error")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, ex, label in zip(axes, ["rho", "u", "p"], [rho_ex_m, u_ex_m, p_ex_m], ["Density", "Velocity", "Pressure"]):
    ax.plot(num_m["x"], num_m[col], label="Numerical (MUSCL+RK2)", linewidth=2)
    ax.plot(num_m["x"], ex, "--", label="Exact", linewidth=2)
    ax.set_xlabel("x"); ax.set_ylabel(label); ax.legend()
plt.suptitle("MUSCL+RK2 scheme vs exact solution")
plt.tight_layout()
plt.show()


## 8. Formal order-of-accuracy verification (smooth test problem)

Sod's shock + contact + rarefaction-fan kinks contaminate a direct
convergence measurement. This test instead advects a smooth density
pulse at constant velocity/pressure — an exact solution of the full
nonlinear system with no discontinuities, so true order of accuracy can
be measured cleanly.

In [ ]:
%%writefile smooth_advection_accuracy_test.cpp
// ============================================================================
// Formal order-of-accuracy verification for the first-order (Rusanov +
// forward Euler) and MUSCL+RK2 schemes.
//
// Why this test, and not the Sod shock tube: the Sod problem always has a
// shock, a contact discontinuity, AND two rarefaction-fan kinks (C0 but not
// C1 continuous) -- every one of these degrades a scheme's LOCAL accuracy,
// contaminating any GLOBAL convergence-rate measurement no matter how
// carefully you mask around the shock/contact (confirmed empirically: masking
// only those two still gave an anomalously low order ~0.9 for MUSCL, because
// the rarefaction-fan kinks were still uncontrolled).
//
// This test instead uses a density pulse advecting at CONSTANT velocity and
// pressure. With u, p spatially and temporally uniform, one can show
// algebraically that all three Euler equations reduce to the same scalar
// advection equation for rho -- so this is an EXACT solution of the full
// nonlinear system, perfectly smooth, no discontinuities anywhere, ever.
// With periodic BC, u=1, domain [0,1], t_final=1.0, the pulse completes
// exactly one full loop: the exact final state equals the INITIAL state,
// so no separate exact-solution code is needed at all.
//
// Result (see conversation / README): first-order empirical order -> ~1
// as N increases (correct). MUSCL+RK2 empirical order -> ~1.65, not a
// clean 2.0 -- this is the well-documented Osher-Chakravarthy accuracy
// barrier: minmod (like all standard TVD limiters) provably drops to
// 1st-order AT SMOOTH EXTREMA (it can't distinguish a smooth peak from a
// discontinuity using only local neighbor differences), and this sine wave
// has exactly two such extrema (peak and trough).
// ============================================================================
#include <cstdio>
#include <cmath>
#include <vector>
#include <algorithm>

const double GAMMA = 1.4;
struct State { double rho, mom, E; };

double velocity(const State& s) { return s.mom / s.rho; }
double pressure(const State& s) { double u=velocity(s); return (GAMMA-1.0)*(s.E-0.5*s.rho*u*u); }
double sound_speed(const State& s) { return std::sqrt(GAMMA*pressure(s)/s.rho); }
State physical_flux(const State& s) { double u=velocity(s),p=pressure(s); return State{s.rho*u,s.rho*u*u+p,u*(s.E+p)}; }
State rusanov_flux(const State& UL, const State& UR) {
    State FL=physical_flux(UL), FR=physical_flux(UR);
    double Smax = std::max(std::fabs(velocity(UL))+sound_speed(UL), std::fabs(velocity(UR))+sound_speed(UR));
    return State{ 0.5*(FL.rho+FR.rho)-0.5*Smax*(UR.rho-UL.rho),
                  0.5*(FL.mom+FR.mom)-0.5*Smax*(UR.mom-UL.mom),
                  0.5*(FL.E+FR.E)    -0.5*Smax*(UR.E-UL.E) };
}
double minmod(double a,double b){ if(a*b<=0.0) return 0.0; return (a>0.0)?std::min(a,b):std::max(a,b); }

State ic(double x, double u0, double p0) {
    double rho = 1.0 + 0.5*std::sin(2*M_PI*x);   // smooth periodic perturbation, no shock ever forms
    return State{ rho, rho*u0, p0/(GAMMA-1.0) + 0.5*rho*u0*u0 };
}

// ---- First order: Rusanov + forward Euler, 1 ghost cell each side ----
double run_first_order(int N) {
    double u0=1.0, p0=1.0, dx=1.0/N, CFL=0.45, t_final=1.0;
    std::vector<State> U(N+2), U0(N+2);
    for (int i=0;i<N+2;++i){ double x=(i-0.5)*dx; U[i]=ic(x,u0,p0); U0[i]=U[i]; }
    double t=0.0;
    std::vector<State> F(N+1), U_new(N+2);
    while (t<t_final) {
        U[0]=U[N]; U[N+1]=U[1];               // periodic
        double Smax=0.0;
        for (int i=1;i<=N;++i) Smax=std::max(Smax, std::fabs(velocity(U[i]))+sound_speed(U[i]));
        double dt=CFL*dx/Smax; if (t+dt>t_final) dt=t_final-t;
        for (int i=0;i<=N;++i) F[i]=rusanov_flux(U[i],U[i+1]);
        for (int i=1;i<=N;++i) {
            U_new[i].rho=U[i].rho-(dt/dx)*(F[i].rho-F[i-1].rho);
            U_new[i].mom=U[i].mom-(dt/dx)*(F[i].mom-F[i-1].mom);
            U_new[i].E  =U[i].E  -(dt/dx)*(F[i].E  -F[i-1].E);
        }
        for (int i=1;i<=N;++i) U[i]=U_new[i];
        t+=dt;
    }
    double err2=0.0;
    for (int i=1;i<=N;++i) err2 += (U[i].rho-U0[i].rho)*(U[i].rho-U0[i].rho);
    return std::sqrt(err2/N);
}

// ---- MUSCL + SSP-RK2, 2 ghost cells each side ----
void compute_L(std::vector<State>& U,int N,double dx,std::vector<State>& slope,std::vector<State>& F,std::vector<State>& L){
    for (int i=1;i<=N+2;++i){
        slope[i].rho=minmod(U[i].rho-U[i-1].rho,U[i+1].rho-U[i].rho);
        slope[i].mom=minmod(U[i].mom-U[i-1].mom,U[i+1].mom-U[i].mom);
        slope[i].E  =minmod(U[i].E  -U[i-1].E,  U[i+1].E  -U[i].E);
    }
    for (int i=1;i<=N+1;++i){
        State UL{U[i].rho+0.5*slope[i].rho, U[i].mom+0.5*slope[i].mom, U[i].E+0.5*slope[i].E};
        State UR{U[i+1].rho-0.5*slope[i+1].rho, U[i+1].mom-0.5*slope[i+1].mom, U[i+1].E-0.5*slope[i+1].E};
        F[i]=rusanov_flux(UL,UR);
    }
    for (int i=2;i<=N+1;++i){
        L[i].rho=-(F[i].rho-F[i-1].rho)/dx;
        L[i].mom=-(F[i].mom-F[i-1].mom)/dx;
        L[i].E  =-(F[i].E  -F[i-1].E)  /dx;
    }
}
double run_muscl_rk2(int N) {
    double u0=1.0, p0=1.0, dx=1.0/N, CFL=0.45, t_final=1.0;
    std::vector<State> U(N+4), U0(N+4);
    for (int i=0;i<N+4;++i){ double x=(i-2.0+0.5)*dx; U[i]=ic(x,u0,p0); U0[i]=U[i]; }
    std::vector<State> slope(N+4), F(N+2), L1(N+4), L2(N+4), U_star(N+4);
    double t=0.0;
    while (t<t_final) {
        U[1]=U[N+1]; U[0]=U[N];  U[N+2]=U[2]; U[N+3]=U[3];  // periodic, 2 ghost cells
        double Smax=0.0;
        for (int i=2;i<=N+1;++i) Smax=std::max(Smax, std::fabs(velocity(U[i]))+sound_speed(U[i]));
        double dt=CFL*dx/Smax; if (t+dt>t_final) dt=t_final-t;

        compute_L(U,N,dx,slope,F,L1);
        U_star=U;
        for (int i=2;i<=N+1;++i){ U_star[i].rho=U[i].rho+dt*L1[i].rho; U_star[i].mom=U[i].mom+dt*L1[i].mom; U_star[i].E=U[i].E+dt*L1[i].E; }

        U_star[1]=U_star[N+1]; U_star[0]=U_star[N]; U_star[N+2]=U_star[2]; U_star[N+3]=U_star[3];
        compute_L(U_star,N,dx,slope,F,L2);
        for (int i=2;i<=N+1;++i){
            double rho2=U_star[i].rho+dt*L2[i].rho, mom2=U_star[i].mom+dt*L2[i].mom, E2=U_star[i].E+dt*L2[i].E;
            U[i].rho=0.5*(U[i].rho+rho2); U[i].mom=0.5*(U[i].mom+mom2); U[i].E=0.5*(U[i].E+E2);
        }
        t+=dt;
    }
    double err2=0.0;
    for (int i=2;i<=N+1;++i) err2 += (U[i].rho-U0[i].rho)*(U[i].rho-U0[i].rho);
    return std::sqrt(err2/N);
}

int main() {
    int Ns[] = {50,100,200,400,800};
    printf("N\tL2_first_order\tL2_MUSCL_RK2\n");
    for (int N : Ns) printf("%d\t%.6e\t%.6e\n", N, run_first_order(N), run_muscl_rk2(N));
    printf("\nCompute empirical order via: order = polyfit(log(1/N), log(L2), 1)[0]\n");
    printf("Expected: first-order -> ~1.0, MUSCL+RK2 -> ~1.6-1.7 (see file header comment for why not exactly 2.0)\n");
    return 0;
}


In [ ]:
!g++ -O3 -o smooth_test smooth_advection_accuracy_test.cpp
!./smooth_test


## 9. Compute empirical order of accuracy from the table above

In [ ]:
import subprocess

result = subprocess.run(["./smooth_test"], capture_output=True, text=True)
lines = result.stdout.strip().split("\n")

Ns, l2_1st, l2_muscl_list = [], [], []
for line in lines[1:]:
    parts = line.split("\t")
    if len(parts) == 3 and parts[0].isdigit():
        Ns.append(int(parts[0]))
        l2_1st.append(float(parts[1]))
        l2_muscl_list.append(float(parts[2]))

Ns = np.array(Ns); l2_1st = np.array(l2_1st); l2_muscl_list = np.array(l2_muscl_list)
order_1st = np.polyfit(np.log(1.0 / Ns), np.log(l2_1st), 1)[0]
order_muscl = np.polyfit(np.log(1.0 / Ns), np.log(l2_muscl_list), 1)[0]

print(f"Empirical order (first-order Rusanov):  {order_1st:.3f}  (expected -> ~1.0)")
print(f"Empirical order (MUSCL + SSP-RK2):      {order_muscl:.3f}  (expected -> ~1.6-1.7, not 2.0 -- Osher-Chakravarthy accuracy barrier at smooth extrema)")

plt.figure(figsize=(6,5))
plt.loglog(1.0/Ns, l2_1st, 'o-', label=f"1st order (slope={order_1st:.2f})")
plt.loglog(1.0/Ns, l2_muscl_list, 's-', label=f"MUSCL+RK2 (slope={order_muscl:.2f})")
plt.xlabel("dx"); plt.ylabel("L2 error"); plt.legend(); plt.title("Order-of-accuracy convergence")
plt.grid(True, which="both", alpha=0.3)
plt.show()


## Status

- Task 1 (exact solution + L2 validation): **done**
- Task 2 (MUSCL + SSP-RK2, CPU, validated + formal order-of-accuracy): **done**
- Next: GPU port of MUSCL+RK2, then Task 3 (2D oblique shock extension)
